# Generating Intelligibility Scores

To compute intelligibility scores, we assume that both the reference (ground truth) and hypothesis (listener transcription) sequences are already normalized using the TextCleaner. This ensures that punctuation, numbers, contractions, and spelling inconsistencies do not affect the scoring.

The scoring process uses a Levenshtein alignment (edit distance) to determine how many words in the hypothesis match the reference. It also incorporates phonemic representations when a pronunciation dictionary is provided, allowing for more robust scoring against homophones or alternative spellings.

## Step-by-Step Explanation

1. Initialization (`__init__`)
    * `self.transformation` defines a series of text preprocessing steps using Jiwer:
        * Remove nonwords (Kaldi-style).
        * Strip leading/trailing spaces.
        * Convert to uppercase for case-insensitive matching.
        * Remove multiple spaces and reduce whitespace.
        * Reduce to a single sentence.
    * `self.transformation_towords` converts sentences to **lists of words**, which is required for the Levenshtein alignment.
    * `pron_dict` is an optional pronunciation dictionary that allows generating **phonetic variants** of words.
  
2. Prepare Word Sequences (`get_word_sequence`)
    * Applies the main transformation pipeline to a sentence.
    * Returns a cleaned sequence of words ready for scoring.
  
3. Scoring (`score`)
    * Accepts `ref` (reference) and `hyp` (hypothesis), which can each be a string or a list of strings.
    * Normalizes each string using the transformation pipeline.
    * If a pronunciation dictionary is provided:
        * Generates phonemic alternatives for both reference and hypothesis.
        * Keeps track of the original hypothesis corresponding to each phonemic form.
    * Creates all **possible pairs** between hypothesis forms and reference forms.

4. Compute Alignment and Hits
    * For each pair of reference and hypothesis sequences, the function uses `process_words` (based on Jiwer) to compute the **number of hits** (correctly matched words).
    * Chooses the pair with the **maximum number of hits** as the best alignment.
    * Optionally prints a visual alignment of the best hypothesis against the reference.

5. Return Metrics
    * `prompt`: the original reference (lowercased)
    * `response`: the best hypothesis matched to the reference (lowercased)
    * `total_words`: number of words in the reference
    * `hits`: number of correctly matched words
    * `correctness`: proportion of correctly matched words (hits / total_words)

In [1]:
import jiwer
import logging
import inflect

from itertools import product
from jiwer import process_words

p = inflect.engine()
logger = logging.getLogger(__name__)


class SentenceScorer:
    def __init__(self, pron_dict=None):
        self.transformation = jiwer.Compose(
            [
                jiwer.RemoveKaldiNonWords(),
                jiwer.Strip(),
                jiwer.ToUpperCase(),
                jiwer.RemoveMultipleSpaces(),
                jiwer.RemoveWhiteSpace(replace_by_space=True),
                jiwer.ReduceToSingleSentence(),
            ]
        )
        self.transformation_towords = jiwer.Compose(
            [jiwer.ReduceToListOfListOfWords(word_delimiter=" ")]
        )

        self.pron_dict = pron_dict

    def get_word_sequence(self, sentence):
        return self.transformation(sentence)

    def lcs_length(self, ref, hyp):
        ref = ref.lower().split()
        hyp = hyp.lower().split()
        n, m = len(ref), len(hyp)
        dp = [[0] * (m + 1) for _ in range(n + 1)]

        for i in range(n):
            for j in range(m):
                if ref[i] == hyp[j]:
                    dp[i + 1][j + 1] = dp[i][j] + 1
                else:
                    dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])
        return dp[n][m]

    def score(self, ref, hyp, show_alignment=False):
        if isinstance(ref, str):
            _ref_form = [ref]
        else:
            _ref_form = ref.copy()

        if isinstance(hyp, str):
            _hyp_forms = [hyp]
        else:
            _hyp_forms = hyp.copy()

        hyp_forms, ref_forms = [], []
        hyp_match_pron_ori= {}
        for hyp_form in _hyp_forms:
            hyp_form = self.transformation(hyp_form)
            hyp_forms.extend(self.pron_dict.get_pronunciations(hyp_form, ref=False))
            for pron in hyp_forms:
                hyp_match_pron_ori[pron] = hyp_form

        for ref_form in _ref_form:
            ref_form = self.transformation(ref_form)
            ref_forms.extend(self.pron_dict.get_pronunciations(ref_form, ref=True))

        alternatives = [(x, y) for x, y in product(hyp_forms, ref_forms)]

        measures = [
            process_words(
                ref,
                hyp,
                reference_transform=self.transformation_towords,
                hypothesis_transform=self.transformation_towords,
            )
            for hyp, ref in alternatives
        ]

        hits = [m.hits for m in measures]
        best_index = hits.index(max(hits))

        if show_alignment:
            print("Alignment:")
            print(jiwer.visualize_alignment(measures[best_index], show_measures=False))



        return {
            "prompt": ref.lower(),
            "response": hyp_match_pron_ori[alternatives[best_index][0]].lower(),
            "total_words": len(measures[best_index].references[0]),
            "hits": measures[best_index].hits,
            "correctness": measures[best_index].hits
            / len(measures[best_index].references[0]),
        }


In [9]:
from IPython.display import display, Markdown
from scorer import SentenceScorer
from text_cleaner import TextCleaner
from pron_dictionary import PronDictionary

1. First, we normalise the sequences

In [14]:
ref = "I don't know if I'll go to the party"
hyp = "I dont know if I will to the party"

In [15]:
# Normalise sequences
cleaner = TextCleaner(
    contractions_file='../input_files/contractions.csv',
    spellings_file='../input_files/spelling_corrections.csv'
)
ref = cleaner(ref, descending=True)[0]
hyp = cleaner(hyp)

display(Markdown(f"**Reference:**\n- `{ref}`"))
display(Markdown("**Hypothesis:**"))
for h in hyp:
    display(Markdown(f"- `{h}`"))

**Reference:**
- `i do not know if i will go to the party`

**Hypothesis:**

- `i don't know if i will to the party`

- `i do not know if i will to the party`

2. Compute the max score from both alternatives

In [16]:
pron_dict = PronDictionary('../input_files/beep-1.0')
scorer = SentenceScorer(pron_dict=pron_dict)

In [18]:
score = scorer.score(ref, hyp, show_alignment=True)

from IPython.display import display, Markdown
display(Markdown(f"**Reference:** `{score['prompt']}`"))
display(Markdown(f"**Hypothesis:** `{score['response']}`"))
display(Markdown(f"**Total words:** {score['total_words']}"))
display(Markdown(f"**Hits:** {score['hits']}"))
display(Markdown(f"**Score:** {score['correctness']:.2f}"))

Alignment:
sentence 1
REF: ay d-uw n-oh-t n-ow ih-f ay w-ih-l g-ow t-ax dh-ax p-aa-t-iy
HYP: ay d-uw n-oh-t n-ow ih-f ay w-ih-l **** t-ax dh-ax p-aa-t-iy
                                           D                     



**Reference:** `i do not know if i will go to the party`

**Hypothesis:** `i do not know if i will to the party`

**Total words:** 11

**Hits:** 10

**Score:** 0.91